# Colab A — Discover + HF upload + Download shard 0/3

**3-Colab схема:** A (этот, shard 0) + B (shard 1) + C (shard 2)

State хранится **локально** на `/content/` и синкается на Drive каждые 2 мин. Видео сразу уходят в HF и удаляются локально без засорения корзины.

> **Runbook:** `Fetcher/docs/COLAB_20K_RUN_RU.md`

## 0. CONFIG

In [ ]:
CONFIG = {
    "repo_url":            "https://github.com/lebedev-ilia/TrendFlowML.git",
    "fetcher":             "/content/TrendFlowML/Fetcher",
    "drive_secrets":       "/content/drive/MyDrive/trendflow_secrets",
    "output_dir":          "/content/dataset_runs/20k-main",       # локально — не засоряет корзину Drive
    "drive_backup_dir":    "/content/drive/MyDrive/dataset_runs/20k-main",
    "hf_repo_prefix":      "Ilialebedev",
    "campaign_profile":    "20k",
    "worker_id":           "colab-a",
    "worker_shard_index":  0,
    "worker_shard_count":  3,
    "parallel_colab_count": 3,
    "discover_limit":      None,   # None = полный; 500 для smoke-теста
    "download_pause_after_success_seconds": 5,
    "download_pause_after_bot_seconds":     120,
    "hf_commit_min_interval_seconds":       95,
    "hf_parallel_colab_count":             3,
    "sync_interval_sec":   120,
    "metrics_discover":    9095,
    "metrics_workers":     9096,
}

import json
from pathlib import Path

OUTPUT = Path(CONFIG["output_dir"])
OUTPUT.mkdir(parents=True, exist_ok=True)

OVERRIDES = {k: CONFIG[k] for k in [
    "worker_id", "worker_shard_index", "worker_shard_count",
    "hf_parallel_colab_count", "hf_commit_min_interval_seconds",
    "download_pause_after_success_seconds", "download_pause_after_bot_seconds",
] if k in CONFIG}
OVERRIDES_PATH = OUTPUT / "config_overrides_a.json"
OVERRIDES_PATH.write_text(json.dumps(OVERRIDES, indent=2), encoding="utf-8")

print("worker_id :", CONFIG["worker_id"])
print("shard     :", CONFIG["worker_shard_index"], "/", CONFIG["worker_shard_count"])
print("output_dir:", OUTPUT)
print("backup_dir:", CONFIG["drive_backup_dir"])


## 1. Drive + git + pip

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import subprocess
from pathlib import Path

dest = Path("/content/TrendFlowML")
if (dest / ".git").exists():
    subprocess.run(["git", "-C", str(dest), "pull"], check=False)
else:
    subprocess.run(["git", "clone", CONFIG["repo_url"], str(dest)], check=True)

# pip install через requirements.txt (Fetcher не имеет pyproject.toml)
subprocess.run(["pip", "install", "-q", "-r",
    str(Path(CONFIG["fetcher"]) / "requirements.txt")], check=True)
subprocess.run(["pip", "install", "-q", "-U",
    "huggingface_hub", "yt-dlp", "pytubefix>=9.5.0",
    "nodejs-wheel-binaries", "google-api-python-client"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "nodejs"], check=False)
print("install done")


## 2. HF_TOKEN + YouTube keys + cookies

`HF_TOKEN` → добавьте в **Colab Secrets** (🔑 на панели слева).

In [ ]:
import os, shutil
from pathlib import Path

# HF_TOKEN — добавьте в Colab Secrets (🔑) или раскомментируйте строку ниже
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN").strip()
except Exception:
    pass
# os.environ["HF_TOKEN"] = "hf_..."

if not os.environ.get("HF_TOKEN", "").startswith("hf_"):
    raise RuntimeError("Добавьте HF_TOKEN в Colab Secrets или впишите вручную")

Path(CONFIG["output_dir"]).joinpath(".hf_token").write_text(
    os.environ["HF_TOKEN"].strip() + "\n", encoding="utf-8")
print("HF_TOKEN OK")

secrets = Path(CONFIG["drive_secrets"])

# YouTube API ключи
keys_dst = Path(CONFIG["fetcher"]) / "fetcher/dataset_collector/keys"
keys_dst.mkdir(parents=True, exist_ok=True)
keys_src = secrets / "keys" / "keys.txt"
if keys_src.is_file():
    shutil.copy2(keys_src, keys_dst / "keys.txt")
    n = len([l for l in keys_src.read_text().splitlines() if l.strip()])
    print(f"YouTube keys: {n} ключей")
else:
    # Раскомментировать и вписать ключи напрямую:
    # KEYS = ["AIzaSy...", "AIzaSy..."]
    # (keys_dst / "keys.txt").write_text("\n".join(KEYS))
    raise FileNotFoundError(f"keys.txt не найден: {keys_src}")

# Cookies
cookie_dst = Path(CONFIG["fetcher"]) / "fetcher/dataset_collector/cookies"
cookie_dst.mkdir(parents=True, exist_ok=True)
cookie_src = secrets / "cookies"
if cookie_src.is_dir():
    for f in cookie_src.glob("*.txt"):
        shutil.copy2(f, cookie_dst / f.name)
    print(f"Cookies: {len(list(cookie_dst.glob('*.txt')))} файлов")
else:
    print("WARN: cookies не найдены на Drive — скачивание может упасть на bot-detection")


## 3. Restore state с Drive (resume после обрыва)

При первом запуске — просто создаёт папки. При resume — подхватывает state.

In [ ]:
import subprocess
from pathlib import Path

backup = Path(CONFIG["drive_backup_dir"])
local  = Path(CONFIG["output_dir"])
local.mkdir(parents=True, exist_ok=True)

restored = []
for d in ["state", "shards", "downloads"]:
    src = backup / d
    if src.is_dir():
        dst = local / d
        dst.mkdir(parents=True, exist_ok=True)
        subprocess.run(["rsync", "-a", "--ignore-existing",
                        str(src) + "/", str(dst) + "/"], check=False)
        restored.append(d)

print("Restored from Drive:", restored if restored else "ничего (первый запуск)")


## 4. Auto-sync state → Drive (каждые 2 мин)

⚠️ Запустите эту ячейку **один раз** — она работает в фоне весь сеанс.

In [ ]:
import threading, subprocess, time
from pathlib import Path

_sync_stop = False

def _do_sync():
    local  = Path(CONFIG["output_dir"])
    backup = Path(CONFIG["drive_backup_dir"])
    backup.mkdir(parents=True, exist_ok=True)
    synced = []
    for d in ["state", "shards", "downloads"]:
        src = local / d
        if not src.is_dir():
            continue
        dst = backup / d
        dst.mkdir(parents=True, exist_ok=True)
        r = subprocess.run(
            ["rsync", "-a",
             "--exclude=*.mp4", "--exclude=*.video.tmp", "--exclude=*.audio.tmp",
             str(src) + "/", str(dst) + "/"],
            capture_output=True
        )
        if r.returncode == 0:
            synced.append(d)
    return synced

def _sync_loop():
    interval = CONFIG["sync_interval_sec"]
    while not _sync_stop:
        time.sleep(interval)
        try:
            synced = _do_sync()
            print(f"[sync {time.strftime('%H:%M:%S')}] → Drive: {synced}")
        except Exception as e:
            print(f"[sync ERROR] {e}")

_t = threading.Thread(target=_sync_loop, daemon=True, name="drive-sync")
_t.start()

print(f"Auto-sync запущен каждые {CONFIG['sync_interval_sec']}s → {CONFIG['drive_backup_dir']}")
print("Принудительный sync: запустите ячейку 9")


## 5. Discover

Запускается в фоне. Для smoke-теста: `discover_limit: 500` в CONFIG.

In [ ]:
import subprocess, os
from pathlib import Path

cmd = [
    "python", "scripts/colab_20k_bootstrap.py",
    "--campaign-profile",   CONFIG["campaign_profile"],
    "--role",               "discover",
    "--output-dir",         CONFIG["output_dir"],
    "--hf-repo-prefix",     CONFIG["hf_repo_prefix"],
    "--worker-shard-index", str(CONFIG["worker_shard_index"]),
    "--worker-shard-count", str(CONFIG["worker_shard_count"]),
    "--parallel-colab-count", str(CONFIG["parallel_colab_count"]),
    "--youtube-keys-file",  "fetcher/dataset_collector/keys/keys.txt",
    "--metrics-port",       str(CONFIG["metrics_discover"]),
    "--config-overrides-json", str(OVERRIDES_PATH),
]
if CONFIG["discover_limit"]:
    cmd += ["--limit", str(CONFIG["discover_limit"])]

log = Path("/content/discover_log.txt")
with open(log, "w") as fh:
    p_discover = subprocess.Popen(cmd, cwd=CONFIG["fetcher"],
                                  stdout=fh, stderr=subprocess.STDOUT,
                                  env=os.environ.copy())
print(f"discover pid={p_discover.pid}  →  !tail -f {log}")


## 6. Upload HF shards (метаданные → HF)

Запустите через 2-3 мин после старта discover.

In [ ]:
# Запускать через ~2-3 минуты после discover (чтобы были shards для upload)
import subprocess, os
from pathlib import Path

cmd = [
    "python", "scripts/colab_20k_bootstrap.py",
    "--campaign-profile",    CONFIG["campaign_profile"],
    "--role",                "workers",
    "--worker-kinds",        "upload-hf-shards",
    "--output-dir",          CONFIG["output_dir"],
    "--hf-repo-prefix",      CONFIG["hf_repo_prefix"],
    "--parallel-colab-count", str(CONFIG["parallel_colab_count"]),
    "--metrics-port",        "0",
    "--config-overrides-json", str(OVERRIDES_PATH),
]
log = Path("/content/shards_log.txt")
with open(log, "w") as fh:
    p_shards = subprocess.Popen(cmd, cwd=CONFIG["fetcher"],
                                stdout=fh, stderr=subprocess.STDOUT,
                                env=os.environ.copy())
print(f"upload-hf-shards pid={p_shards.pid}  →  !tail -f {log}")


## 7. Download workers shard 0/3 (опционально)

Можно пропустить — B и C скачают без A.

In [ ]:
# Опционально — download shard 0/3 прямо на Colab A
# Пропустите если A нагружена: B и C скачают всё
import subprocess, os
from pathlib import Path

cmd = [
    "python", "scripts/colab_20k_bootstrap.py",
    "--campaign-profile",    CONFIG["campaign_profile"],
    "--role",                "workers-download",
    "--output-dir",          CONFIG["output_dir"],
    "--hf-repo-prefix",      CONFIG["hf_repo_prefix"],
    "--worker-id",           CONFIG["worker_id"],
    "--worker-shard-index",  str(CONFIG["worker_shard_index"]),
    "--worker-shard-count",  str(CONFIG["worker_shard_count"]),
    "--parallel-colab-count", str(CONFIG["parallel_colab_count"]),
    "--cookie-files-dir",    "fetcher/dataset_collector/cookies",
    "--metrics-port",        str(CONFIG["metrics_workers"]),
    "--config-overrides-json", str(OVERRIDES_PATH),
]
log = Path("/content/download_a_log.txt")
with open(log, "w") as fh:
    p_dl = subprocess.Popen(cmd, cwd=CONFIG["fetcher"],
                            stdout=fh, stderr=subprocess.STDOUT,
                            env=os.environ.copy())
print(f"workers-download pid={p_dl.pid} shard=0/3  →  !tail -f {log}")


## 8. Статус

In [ ]:
import subprocess, os
from pathlib import Path

subprocess.run([
    "python", "scripts/colab_20k_bootstrap.py",
    "--campaign-profile", CONFIG["campaign_profile"],
    "--role",             "status",
    "--output-dir",       CONFIG["output_dir"],
], cwd=CONFIG["fetcher"], env=os.environ.copy(), check=False)


## 9. Принудительный sync

In [ ]:
# Принудительный sync в Drive прямо сейчас
import time
synced = _do_sync()
print(f"[{time.strftime('%H:%M:%S')}] Sync → Drive: {synced}")
